# Assignment 2 — Q2. Diabetes Sampling
**Course:** CS5530 — Principles of Data Science

**Student:** Kai Son — 16401784

I use all 768 rows as the population. I keep the original values, including zeros, because this question compares samples with the supplied population. Some zeros may not be valid medical measurements. This is a sampling exercise.

In [ ]:
# Q2 — Load the dataset
import pandas as pd
import matplotlib.pyplot as plt
import random
from google.colab import drive

drive.mount("/content/drive")

file_path = "/content/drive/MyDrive/Colab Notebooks/CS-5530/Assignment 2 & 3/diabetes.csv"

diabetes = pd.read_csv(file_path)

print("Rows and columns:", diabetes.shape)
display(diabetes.head())


## Q2(a). Sample 25 observations

I use seed 1784 to get the same sample each time. I select 25 different rows without replacement, then compare their glucose mean and maximum with the population.

In [ ]:
# Q2(a) — Draw a random sample of 25 rows
seed = 1784
rng = random.Random(seed)
sample_indices = rng.sample(list(diabetes.index), 25)
sample = diabetes.loc[sample_indices].copy()

glucose_comparison = pd.DataFrame({
    'Population': [diabetes['Glucose'].mean(), diabetes['Glucose'].max()],
    'Sample': [sample['Glucose'].mean(), sample['Glucose'].max()]
}, index=['Mean', 'Maximum'])

display(glucose_comparison.round(2))


In [ ]:
# Q2(a) — Plot glucose comparisons
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, statistic in zip(axes, ['Mean', 'Maximum']):
    glucose_comparison.loc[statistic].plot.bar(
        ax=ax, color=['steelblue', 'orange'], rot=0
    )
    ax.set_title('Glucose ' + statistic)
    ax.set_ylabel('Glucose')
    ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print(f"Population mean: {diabetes['Glucose'].mean():.2f}")
print(f"Sample mean: {sample['Glucose'].mean():.2f}")
print(f"Population maximum: {diabetes['Glucose'].max():.0f}")
print(f"Sample maximum: {sample['Glucose'].max():.0f}")


The sample can differ from the population because it contains only 25 rows. A random sample does not necessarily include the largest value.

## Q2(b). Compare the 98th percentile of BMI

The 98th percentile is a value near the upper end of the data. I calculate it for the same 25-row sample and the population. Pandas uses linear interpolation by default.

In [ ]:
# Q2(b) — Calculate and plot the BMI 98th percentile
bmi_comparison = pd.DataFrame({
    'Population': [diabetes['BMI'].quantile(0.98)],
    'Sample': [sample['BMI'].quantile(0.98)]
}, index=['98th percentile'])

display(bmi_comparison.round(2))
bmi_comparison.loc['98th percentile'].plot.bar(
    color=['steelblue', 'orange'], rot=0, figsize=(6, 4)
)
plt.title('BMI 98th percentile')
plt.ylabel('BMI')
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()


## Q2(c). Bootstrap BloodPressure

I draw 500 samples of 150 rows with replacement from the full population. With replacement means the same row can appear more than once. For each sample, I calculate the mean, sample standard deviation, and 98th percentile of BloodPressure. I use the 98th percentile because part c does not specify a percentile.

In [ ]:
# Q2(c) — Generate 500 bootstrap samples
rng = random.Random(seed)
bootstrap_results = []

for i in range(500):
    indices = rng.choices(list(diabetes.index), k=150)
    blood_pressure = diabetes.loc[indices, 'BloodPressure']
    bootstrap_results.append({
        'Mean': blood_pressure.mean(),
        'Standard deviation': blood_pressure.std(ddof=1),
        '98th percentile': blood_pressure.quantile(0.98)
    })

bootstrap_results = pd.DataFrame(bootstrap_results)

# ddof=0 describes the full population; ddof=1 describes each sample.
population_stats = pd.Series({
    'Mean': diabetes['BloodPressure'].mean(),
    'Standard deviation': diabetes['BloodPressure'].std(ddof=0),
    '98th percentile': diabetes['BloodPressure'].quantile(0.98)
})

blood_pressure_comparison = pd.DataFrame({
    'Population': population_stats,
    'Bootstrap average': bootstrap_results.mean()
})
blood_pressure_comparison['Difference'] = (
    blood_pressure_comparison['Bootstrap average']
    - blood_pressure_comparison['Population']
)
display(blood_pressure_comparison.round(2))


In [ ]:
# Q2(c) — Plot population and average bootstrap statistics
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, statistic in zip(axes, blood_pressure_comparison.index):
    blood_pressure_comparison.loc[
        statistic, ['Population', 'Bootstrap average']
    ].plot.bar(ax=ax, color=['steelblue', 'orange'], rot=0)
    ax.set_title(statistic)
    ax.set_ylabel('BloodPressure')
    ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

for statistic, row in blood_pressure_comparison.iterrows():
    print(
        f"{statistic}: population = {row['Population']:.2f}, "
        f"bootstrap average = {row['Bootstrap average']:.2f}, "
        f"difference = {row['Difference']:.2f}"
    )


I compare the average of each statistic across 500 samples with its population value. The average sample standard deviation is not the standard error of the bootstrap means. The results may be close without being exactly equal.

In [ ]:
# Q2 — Save the final dataset
import os

# Keep all 768 rows for the population comparison.
output_folder = '/content/drive/MyDrive/Colab Notebooks/CS-5530/Assignment 2 & 3/Q2_Diabetes/data_clean'
os.makedirs(output_folder, exist_ok=True)
diabetes.to_csv(output_folder + '/diabetes_final.csv', index=False)
